In [ ]:
# ====== Strava App credentials ======
CLIENT_ID = "199191"
CLIENT_SECRET = "d0905234a2008733278d3320e274021c5b107be4"

# 依照官方 getting-started 範例使用 localhost callback domain。:contentReference[oaicite:3]{index=3}
REDIRECT_URI = "http://localhost/exchange_token"

BASE_URL = "https://www.strava.com"
API_URL  = "https://www.strava.com/api/v3"


In [ ]:
from urllib.parse import urlencode

params = {
    "client_id": CLIENT_ID,
    "response_type": "code",
    "redirect_uri": REDIRECT_URI,
    "approval_prompt": "force",
    # 依你的需求：抓活動建議 read + activity:read_all
    "scope": "read,activity:read_all",
}

auth_url = f"{BASE_URL}/oauth/authorize?{urlencode(params)}"
auth_url


In [ ]:
import requests

AUTHORIZATION_CODE = "b21079f63656e18a13fce578f2bcf333e50db38d"

token_resp = requests.post(
    f"{BASE_URL}/oauth/token",
    data={
        "client_id": CLIENT_ID,
        "client_secret": CLIENT_SECRET,
        "code": AUTHORIZATION_CODE,
        "grant_type": "authorization_code",
    },
    timeout=30,
)
token_resp.raise_for_status()
token_data = token_resp.json()
token_data

ACCESS_TOKEN  = token_data["access_token"]
REFRESH_TOKEN = token_data["refresh_token"]
EXPIRES_AT    = int(token_data["expires_at"])  # epoch seconds

ACCESS_TOKEN, REFRESH_TOKEN, EXPIRES_AT

In [ ]:
def strava_get(path, params=None):
    headers = {"Authorization": f"Bearer {ACCESS_TOKEN}"}
    r = requests.get(f"{API_URL}{path}", headers=headers, params=params or {}, timeout=30)
    r.raise_for_status()
    return r.json()

me = strava_get("/athlete")
me


In [ ]:
def fetch_activity_detail(access_token, run_id):
    """Fetch detailed activity data from Strava for a given activity id."""
    global ACCESS_TOKEN
    ACCESS_TOKEN = access_token
    return strava_get(f"/activities/{run_id}")


In [ ]:
def fetch_activities(per_page=50, pages=3, after=None):
    all_acts = []
    for page in range(1, pages + 1):
        params = {"per_page": per_page, "page": page}
        if after is not None:
            params["after"] = int(after)
        batch = strava_get("/athlete/activities", params=params)
        if not batch:
            break
        all_acts.extend(batch)
    return all_acts

acts = fetch_activities(per_page=50, pages=5)
len(acts), acts[0].get("type"), acts[0].get("start_date")


In [ ]:
import time

def refresh_access_token_if_needed():
    global ACCESS_TOKEN, REFRESH_TOKEN, EXPIRES_AT

    now = int(time.time())
    if now < EXPIRES_AT - 120:
        return  # 還沒快到期

    resp = requests.post(
        f"{BASE_URL}/oauth/token",
        data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "refresh_token",
            "refresh_token": REFRESH_TOKEN,
        },
        timeout=30,
    )
    resp.raise_for_status()
    data = resp.json()
    ACCESS_TOKEN  = data["access_token"]
    REFRESH_TOKEN = data["refresh_token"]
    EXPIRES_AT    = int(data["expires_at"])

# 之後每次要抓資料前先呼叫一次
refresh_access_token_if_needed()


In [ ]:
# =========================
# Gear (Shoes) helpers
# =========================
GEAR_CACHE = {}

def fetch_gear_name(gear_id: str) -> str:
    """Return shoe/gear name for Strava gear_id. Uses cache to reduce API calls."""
    gear_id = (gear_id or "").strip()
    if not gear_id:
        return ""

    if gear_id in GEAR_CACHE:
        return GEAR_CACHE[gear_id]

    # ensure ACCESS_TOKEN is valid
    refresh_access_token_if_needed()
    data = strava_get(f"/gear/{gear_id}")
    name = (data.get("name") or "").strip()
    GEAR_CACHE[gear_id] = name
    return name


In [ ]:
def sec_to_mmss(sec):
    if sec is None:
        return ""
    m, s = divmod(int(round(sec)), 60)
    return f"{m:02d}:{s:02d}"

def sec_to_hhmmss(sec):
    if sec is None:
        return ""
    sec = int(round(sec))
    h, rem = divmod(sec, 3600)
    m, s = divmod(rem, 60)
    return f"{h:02d}:{m:02d}:{s:02d}"

from datetime import datetime, timedelta

def _parse_iso(s: str):
    # Strava timestamps are ISO 8601; sometimes end with 'Z'
    return datetime.fromisoformat(s.replace("Z", ""))

def start_dt_local(activity: dict):
    # Prefer local timestamp, fallback to UTC timestamp if local missing
    if activity.get("start_date_local"):
        return _parse_iso(activity["start_date_local"])
    if activity.get("start_date"):
        return _parse_iso(activity["start_date"])
    return None

def format_dt(dt: datetime):
    return dt.strftime("%Y-%m-%d %H:%M:%S") if dt else ""

def end_dt_local(activity: dict):
    dt0 = start_dt_local(activity)
    elapsed = activity.get("elapsed_time")
    if dt0 and elapsed is not None:
        return dt0 + timedelta(seconds=int(elapsed))
    return None


In [ ]:
import pandas as pd

def runs_to_df(activities):
    rows = []
    for a in activities:
        if a.get("type") != "Run":
            continue

        dist_km = (a.get("distance") or 0) / 1000.0
        moving_sec = a.get("moving_time") or 0
        if dist_km <= 0:
            continue

        # datetime
        sdt = start_dt_local(a)
        edt = end_dt_local(a)

        # pace (sec/km)
        avg_pace_sec = (moving_sec / dist_km) if dist_km > 0 else None

        gear_id = a.get("gear_id")
        shoes = fetch_gear_name(gear_id) if gear_id else ""


        rows.append({
            "run_id": a.get("id"),
            "name": a.get("name"),
            "type": a.get("type"),
            "sport_type": a.get("sport_type"),
            "start_datetime_local": format_dt(sdt),
            "end_datetime_local": format_dt(edt),
            "distance_km": round(dist_km, 2),
            "moving_time_min_str": sec_to_hhmmss(moving_sec),
            "avg_pace_str": sec_to_mmss(avg_pace_sec),
            "shoes": shoes,
        })

    cols = [
        "run_id","name","type","sport_type",
        "start_datetime_local","end_datetime_local",
        "distance_km","moving_time_min_str","avg_pace_str",
        "shoes"
    ]
    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


In [ ]:
import pandas as pd

def splits_to_df(activity_detail):
    """Build per-km splits table from Strava DetailedActivity.

    Uses DetailedActivity.splits_metric (for metric units). Cadence is NOT included.
    """
    cols = [
        "run_id","name","type","sport_type",
        "start_datetime_local","end_datetime_local",
        "moving_time_min_str","distance_km",
        "distance","km","pace_str","shoes",
    ]

    splits = activity_detail.get("splits_metric") or []
    if not splits:
        return pd.DataFrame(columns=cols)

    run_id = activity_detail.get("id")
    name = activity_detail.get("name")
    type_ = activity_detail.get("type")
    sport_type = activity_detail.get("sport_type")

    sdt = start_dt_local(activity_detail)
    edt = end_dt_local(activity_detail)
    moving_sec = activity_detail.get("moving_time") or 0
    total_dist_km = (activity_detail.get("distance") or 0) / 1000.0

    gear_id = activity_detail.get("gear_id")
    shoes = fetch_gear_name(gear_id) if gear_id else ""

    rows = []
    for sp in splits:
        split_idx = sp.get("split")
        sp_dist_km = (sp.get("distance") or 0) / 1000.0
        sp_moving = sp.get("moving_time") or sp.get("elapsed_time") or 0
        if sp_dist_km <= 0:
            continue
        pace_sec = sp_moving / sp_dist_km

        rows.append({
            "run_id": run_id,
            "name": name,
            "type": type_,
            "sport_type": sport_type,
            "start_datetime_local": format_dt(sdt),
            "end_datetime_local": format_dt(edt),
            "moving_time_min_str": sec_to_hhmmss(moving_sec),
            "distance_km": round(total_dist_km, 2),
            "distance": round(sp_dist_km, 3),
            "km": int(split_idx) if isinstance(split_idx, (int, float)) else split_idx,
            "pace_str": sec_to_mmss(pace_sec),
            "shoes": shoes,
        })

    df = pd.DataFrame(rows)
    for c in cols:
        if c not in df.columns:
            df[c] = None
    return df[cols]


In [ ]:
import time
import pandas as pd

# 先抓 Run 活動列表（摘要）
acts = fetch_activities(per_page=100, pages=5)

# 建立 runs_df（不含踏頻）
runs_df = runs_to_df(acts)

all_splits = []

for run_id in runs_df["run_id"].dropna().astype(int).tolist():
    refresh_access_token_if_needed()

    # 取單一活動詳細資料（包含 splits_metric）
    detail = fetch_activity_detail(ACCESS_TOKEN, run_id)

    # 由 splits_metric 產生每公里 splits_df（不含踏頻）
    split_df = splits_to_df(detail)
    if not split_df.empty:
        all_splits.append(split_df)

    # 小睡一下避免打到 rate limit（可視情況調整/移除）
    time.sleep(0.2)

splits_df = pd.concat(all_splits, ignore_index=True) if all_splits else pd.DataFrame(
    columns=[
        "run_id","name","type","sport_type",
        "start_datetime_local","end_datetime_local",
        "moving_time_min_str","distance_km",
        "distance","km","pace_str"
    ]
)

# 輸出 CSV
runs_df.to_csv("runs.csv", index=False, encoding="utf-8-sig")
splits_df.to_csv("run_splits.csv", index=False, encoding="utf-8-sig")

runs_df.head(), splits_df.head()


In [ ]:
# （已在上一個 cell 產生 runs_df 與 splits_df，並輸出 runs.csv / run_splits.csv）
# 如果你要把資料寫到 Google Sheet，請看下一個 cell。

In [ ]:
import pandas as pd
import gspread
from google.oauth2.service_account import Credentials

# ====== 你要填的 ======
SERVICE_ACCOUNT_JSON = "edwards-running-log-23196fb4f132.json"
SPREADSHEET_ID = "1sDvgxaFUA9i1a0VO61mG6gvZhAhZBNmW4fqTFcEH-qc"  # 不是整段網址，只要 ID
RUNS_SHEET_NAME = "runs"
SPLITS_SHEET_NAME = "run_splits"


def get_gspread_client():
    scopes = [
        "https://www.googleapis.com/auth/spreadsheets",
        "https://www.googleapis.com/auth/drive",
    ]
    creds = Credentials.from_service_account_file(SERVICE_ACCOUNT_JSON, scopes=scopes)
    return gspread.authorize(creds)


def ensure_worksheet(ss, title, rows=1000, cols=30):
    try:
        ws = ss.worksheet(title)
    except gspread.WorksheetNotFound:
        ws = ss.add_worksheet(title=title, rows=str(rows), cols=str(cols))
    return ws


def write_df_overwrite(ws, df: pd.DataFrame):
    # 讓 NaN 變空字串，避免 Google Sheets 顯示 nan
    df2 = df.copy()
    df2 = df2.where(pd.notnull(df2), "")

    # 清空整張表
    ws.clear()

    # 寫入 header + rows
    values = [df2.columns.tolist()] + df2.astype(str).values.tolist()

    # 批次更新（效率更好）
    ws.update(values, value_input_option="USER_ENTERED")


# ====== 使用範例 ======
gc = get_gspread_client()
ss = gc.open_by_key(SPREADSHEET_ID)

# runs_df / splits_df 是你前面抓 Strava 後做好的 DataFrame
# runs_df: 每次跑步一列
# splits_df: 每公里 split 一列

ws_runs = ensure_worksheet(ss, RUNS_SHEET_NAME, rows=max(1000, len(runs_df)+10), cols=max(20, runs_df.shape[1]+5))
ws_splits = ensure_worksheet(ss, SPLITS_SHEET_NAME, rows=max(3000, len(splits_df)+10), cols=max(30, splits_df.shape[1]+5))

write_df_overwrite(ws_runs, runs_df)
write_df_overwrite(ws_splits, splits_df)

print("✅ Wrote runs_df and splits_df to Google Sheets (overwrite).")
